# 1. Load Bronze Table

In [0]:
# Load Bronze table
df_bronze = spark.table("bronze_operational_reporting")
df_bronze.show(5)


+----------+----------+-------------------+-------------------+----------+---------------+-----------+--------+
|Request_ID|Department|         Start_Time|           End_Time|SLA_Status|Processing_Time|Employee_ID|Priority|
+----------+----------+-------------------+-------------------+----------+---------------+-----------+--------+
|  REQ00001|   Finance|2026-06-14 03:42:00|               NULL|  Breached|           NULL|       E353|     Low|
|  REQ00002|Operations|2026-06-05 14:23:00|               NULL|  Breached|           NULL|       E424|     Low|
|  REQ00003|        IT|2026-06-13 12:46:00|2026-06-13 19:32:00|  Breached|          406.0|       E363|     Low|
|  REQ00004|        HR|2026-06-27 07:40:00|               NULL|       Met|           NULL|       E498|    High|
|  REQ00005|   Finance|2026-06-26 19:49:00|               NULL|       Met|           NULL|       E129|    High|
+----------+----------+-------------------+-------------------+----------+---------------+-----------+--

### Add Flag Column Without Overwriting

In [0]:
from pyspark.sql.functions import col, when

df_silver = df_bronze.withColumn(
    "Invalid_Timestamp_Flag",
    when(col("Start_Time").isNull() | col("End_Time").isNull() | (col("End_Time") < col("Start_Time")), 1).otherwise(0)
)


# 2: Handle Missing Values

#1. Request_ID
Critical identifier → rows missing this should be dropped.

In [0]:
from pyspark.sql.functions import col, when


In [0]:
# Drop rows where Request_ID is missing
df_silver = df_bronze.filter(col("Request_ID").isNotNull())


#2: Department Cleaning


If Department is missing, tag as "Unknown". \
This ensures no nulls propagate into dimensional tables.

In [0]:
from pyspark.sql.functions import col, when

df_silver = df_silver.withColumn(
    "Department",
    when(col("Department").isNull(), "Unknown").otherwise(col("Department"))
)


### Standardize Values
In real data, departments may appear with inconsistent casing or typos (hr, FINANCE, It).\
Normalize to title case for consistency.

In [0]:
from pyspark.sql.functions import initcap

df_silver = df_silver.withColumn("Department", initcap(col("Department")))


### Audit Logging
Always log how many rows were affected. This mirrors enterprise practice.

In [0]:
null_dept_count = df_bronze.filter(col("Department").isNull()).count()
print("Rows with null Department replaced:", null_dept_count)


Rows with null Department replaced: 0


# 3: Start_Time / End_Time Cleaning

### Identify Nulls
If either timestamp is missing, the row cannot be used for SLA calculations. \
Strategy: keep the row but tag it as "Invalid_Timestamp" for audit.

In [0]:
from pyspark.sql.functions import when

df_silver = df_silver.withColumn(
    "Start_Time",
    when(col("Start_Time").isNull(), "Invalid_Timestamp").otherwise(col("Start_Time"))
).withColumn(
    "End_Time",
    when(col("End_Time").isNull(), "Invalid_Timestamp").otherwise(col("End_Time"))
)


### Validate Chronology
End_Time must be greater than Start_Time.\
If not, flag as "Invalid_Timestamp".

In [0]:
df_silver = df_silver.withColumn(
    "End_Time",
    when(col("End_Time") < col("Start_Time"), "Invalid_Timestamp").otherwise(col("End_Time"))
)


### Add a Flag Column

In [0]:
from pyspark.sql.functions import col, when

df_silver = df_silver.withColumn(
    "Invalid_Timestamp_Flag",
    when(col("Start_Time").isNull() | col("End_Time").isNull() | (col("End_Time") < col("Start_Time")), 1).otherwise(0)
)


### Audit Logging
Count how many rows were flagged.

In [0]:
invalid_time_count = df_silver.filter(col("Invalid_Timestamp_Flag") == 1).count()
print("Rows flagged with invalid timestamps:", invalid_time_count)


Rows flagged with invalid timestamps: 3322


# 4: SLA_Status Cleaning

### Handle Nulls
Replace missing values with "Unknown". \
This ensures no nulls propagate into compliance metrics.

In [0]:
df_silver = df_silver.withColumn(
    "SLA_Status",
    when(col("SLA_Status").isNull(), "Unknown").otherwise(col("SLA_Status"))
)


### Standardize Values
In messy data, SLA values may appear as "met", "breach", "BREACHED".\
Normalize to consistent title case: "Met", "Breached", "Unknown".

In [0]:
from pyspark.sql.functions import initcap

df_silver = df_silver.withColumn("SLA_Status", initcap(col("SLA_Status")))


### Validate Allowed Values
Only three valid statuses should exist: "Met", "Breached", "Unknown".\
Any other values are forced to "Unknown".

In [0]:
df_silver = df_silver.withColumn(
    "SLA_Status",
    when(col("SLA_Status").isin("Met","Breached","Unknown"), col("SLA_Status")).otherwise("Unknown")
)


### Audit Logging
Count how many rows were corrected.

In [0]:
invalid_sla_count = df_silver.filter(~col("SLA_Status").isin("Met","Breached","Unknown")).count()
print("Rows corrected in SLA_Status:", invalid_sla_count)


Rows corrected in SLA_Status: 0


# 5: Processing_Time Cleaning

### Handle Nulls
If Processing_Time is missing, set it to -1 (flag for invalid).\
This ensures downstream aggregations don’t silently ignore bad data.

In [0]:
df_silver = df_silver.withColumn(
    "Processing_Time",
    when(col("Processing_Time").isNull(), -1).otherwise(col("Processing_Time"))
)


### Validate Range
Negative values or unrealistic durations (e.g., > 1000 minutes) should be flagged as -1.

In [0]:
df_silver = df_silver.withColumn(
    "Processing_Time",
    when((col("Processing_Time") < 0) | (col("Processing_Time") > 1000), -1).otherwise(col("Processing_Time"))
)


### Recompute from Timestamps
For rows with valid Start_Time and End_Time, recompute Processing_Time.\
Use unix_timestamp to calculate duration in minutes.

In [0]:
from pyspark.sql.functions import unix_timestamp

df_silver = df_silver.withColumn(
    "Processing_Time_Recomputed",
    when(col("Invalid_Timestamp_Flag") == 0,
         (unix_timestamp(col("End_Time")) - unix_timestamp(col("Start_Time"))) / 60
    ).otherwise(-1)
)


### Audit Logging
Count how many rows were corrected or recomputed.

In [0]:
invalid_proc_count = df_silver.filter(col("Processing_Time") == -1).count()
print("Rows flagged with invalid Processing_Time:", invalid_proc_count)


Rows flagged with invalid Processing_Time: 3322


# 2.6: Employee_ID Cleaning

### Handle Nulls
If Employee_ID is missing, tag as "Unknown". \
This prevents broken joins when building dimension tables later.

In [0]:
df_silver = df_silver.withColumn(
    "Employee_ID",
    when(col("Employee_ID").isNull(), "Unknown").otherwise(col("Employee_ID"))
)


### Standardize Format
In real data, IDs may have inconsistent casing or whitespace.\
Normalize by trimming spaces and forcing uppercase.

In [0]:
from pyspark.sql.functions import upper, trim

df_silver = df_silver.withColumn("Employee_ID", upper(trim(col("Employee_ID"))))


### Validate Referential Integrity
In enterprise pipelines, Employee_ID should match a master HR dimension.\
For now, simulate by flagging IDs that don’t follow a pattern (e.g., must start with EMP).

In [0]:
df_silver = df_silver.withColumn(
    "Invalid_Employee_Flag",
    when(~col("Employee_ID").rlike("^EMP[0-9]+$"), 1).otherwise(0)
)


### Audit Logging
Count how many rows were corrected or flagged.

In [0]:
invalid_emp_count = df_silver.filter(col("Invalid_Employee_Flag") == 1).count()
print("Rows flagged with invalid Employee_ID:", invalid_emp_count)


Rows flagged with invalid Employee_ID: 10000


# 2.7: Priority Cleaning

### Handle Nulls
Replace missing values with "Unknown".\
Ensures no nulls propagate into KPI dashboards.

In [0]:
df_silver = df_silver.withColumn(
    "Priority",
    when(col("Priority").isNull(), "Unknown").otherwise(col("Priority"))
)


### Standardize Values
Real data often has inconsistent casing (high, LOW, Med).\
Normalize to title case: "High", "Medium", "Low", "Unknown".

In [0]:
from pyspark.sql.functions import initcap

df_silver = df_silver.withColumn("Priority", initcap(col("Priority")))


### Validate Allowed Values
Only four valid values should exist: "High", "Medium", "Low", "Unknown".\
Any other values are forced to "Unknown".

In [0]:
df_silver = df_silver.withColumn(
    "Priority",
    when(col("Priority").isin("High","Medium","Low","Unknown"), col("Priority")).otherwise("Unknown")
)


### Audit Logging
Count how many rows were corrected.

In [0]:
invalid_priority_count = df_silver.filter(~col("Priority").isin("High","Medium","Low","Unknown")).count()
print("Rows corrected in Priority:", invalid_priority_count)


Rows corrected in Priority: 0


# 2.8: Save Silver Table

### 1. Persist Silver Table
Use saveAsTable() to store the cleaned dataset as a managed Delta table:

In [0]:
df_silver.write.format("delta").mode("overwrite").saveAsTable("silver_operational_reporting")

# This ensures the table is stored in the metastore, accessible via both PySpark and SQL.

### . Verify Table Creation
Run a quick query to confirm:

In [0]:
spark.sql("SELECT * FROM silver_operational_reporting LIMIT 10").show()


+----------+----------+-------------------+-------------------+----------+---------------+-----------+--------+----------------------+--------------------------+---------------------+
|Request_ID|Department|         Start_Time|           End_Time|SLA_Status|Processing_Time|Employee_ID|Priority|Invalid_Timestamp_Flag|Processing_Time_Recomputed|Invalid_Employee_Flag|
+----------+----------+-------------------+-------------------+----------+---------------+-----------+--------+----------------------+--------------------------+---------------------+
|  REQ00001|   Finance|2026-06-14 03:42:00|               NULL|  Breached|           -1.0|       E353|     Low|                     1|                      -1.0|                    1|
|  REQ00002|Operations|2026-06-05 14:23:00|               NULL|  Breached|           -1.0|       E424|     Low|                     1|                      -1.0|                    1|
|  REQ00003|        IT|2026-06-13 12:46:00|2026-06-13 19:32:00|  Breached|      

### Audit Row Counts
Compare Bronze vs Silver row counts to ensure no unintended data loss:

In [0]:
bronze_count = spark.table("bronze_operational_reporting").count()
silver_count = spark.table("silver_operational_reporting").count()

print("Bronze row count:", bronze_count)
print("Silver row count:", silver_count)


Bronze row count: 10000
Silver row count: 10000


### Schema Validation
Confirm that all cleaned columns exist and have correct types:

In [0]:
df_silver.printSchema()


root
 |-- Request_ID: string (nullable = true)
 |-- Department: string (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- SLA_Status: string (nullable = true)
 |-- Processing_Time: double (nullable = true)
 |-- Employee_ID: string (nullable = true)
 |-- Priority: string (nullable = true)
 |-- Invalid_Timestamp_Flag: integer (nullable = false)
 |-- Processing_Time_Recomputed: double (nullable = true)
 |-- Invalid_Employee_Flag: integer (nullable = false)

